
# Assignment 2.2 — Advanced Search Problems (OSMnx + NetworkX)

**Goal:** Find the shortest driving-distance route between **Oracle Park** and the **Palace of Fine Arts** in San Francisco using **OSMnx** to load the street network and **NetworkX** to run search algorithms.

The assignment requires:  
1) At least **two search algorithms**, with justification.  
2) A **reflection** on improvements (3–4 paragraphs).  
3) Submission as both **.ipynb** and **.html** with a clear, step-by-step flow. fileciteturn1file8

---

## Conceptual framing (state-space graph)

We model the road network as a weighted directed graph:

- **Nodes**: intersections / road endpoints  
- **Edges**: directed road segments (one-way constraints preserved)  
- **Edge weight**: segment length in meters (`length`)

A route is a path \(p = \langle n_0, n_1, \dots, n_k \rangle\) and its cost is

\[
\text{cost}(p) = \sum_{i=1}^{k} \text{cost}\big(\langle n_{i-1}, n_i\rangle\big)
\]

With nonnegative edge weights, **Dijkstra / uniform-cost search** returns an optimal path (minimum total distance). fileciteturn1file1

For **A\*** we use
\[
f(p) = g(p) + h(p),
\]
where \(g(p)=\text{cost}(p)\) is distance traveled so far and \(h(\cdot)\) estimates remaining distance-to-go. With an **admissible** heuristic (never overestimates), A\* returns an optimal solution while typically expanding fewer paths than uniform-cost search. fileciteturn1file3turn1file4


In [24]:

# =========================
# 0) Setup / Installation
# =========================
# If you run this notebook in Colab or a fresh environment, install OSMnx first.
# (In many local setups, OSMnx may already be installed.)

# Uncomment the next line if needed:
!pip -q install osmnx
!pip -q install scikit-learn

import os
import math
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt

ox.settings.use_cache = True
ox.settings.log_console = True

print("OSMnx version:", ox.__version__)
print("NetworkX version:", nx.__version__)


OSMnx version: 2.0.7
NetworkX version: 3.4.2



## 1) Define the problem (start/goal)

We want the shortest driving route from:

- **Oracle Park** (San Francisco Giants)
- **Palace of Fine Arts**

We represent each place by a latitude/longitude pair.

**Notes:**
- The search will be run on a **drive** network type, so edges respect one-way streets.
- We will search by **distance**, so the edge weight is road segment length in meters.


In [13]:

# =========================
# 1) Coordinates
# =========================
# Coordinates are approximate but sufficient for node-matching on the street graph.

oracle_park = (37.778595, -122.389270)       # (lat, lon)
palace_fine_arts = (37.802376, -122.448210)  # (lat, lon)

print("Oracle Park:", oracle_park)
print("Palace of Fine Arts:", palace_fine_arts)


Oracle Park: (37.778595, -122.38927)
Palace of Fine Arts: (37.802376, -122.44821)



## 2) Load the road network with OSMnx

We download the driving network for San Francisco. Two common choices:

1. `graph_from_place("San Francisco, California, USA", network_type="drive")`  
2. `graph_from_point(center_point, dist=..., network_type="drive")`

We use `graph_from_place` for clarity, then project to a planar CRS so Euclidean distance
makes sense for the A\* heuristic.


In [14]:

# =========================
# 2) Download and project graph
# =========================
place_name = "San Francisco, California, USA"
G = ox.graph_from_place(place_name, network_type="drive")

# Project to a local metric CRS (UTM) for consistent distance calculations.
G_proj = ox.project_graph(G)

print("Original graph:", G)
print("Projected graph:", G_proj)


Original graph: MultiDiGraph with 10035 nodes and 27636 edges
Projected graph: MultiDiGraph with 10035 nodes and 27636 edges



## 3) Map our coordinates to nearest graph nodes

OSMnx provides utilities to find nearest nodes in a graph to a given point.
We compute the nearest node to each landmark (after projection).


In [27]:

# =========================
# 3) Nearest nodes (FIXED)
# =========================

# OSMnx expects longitude (X) and latitude (Y)

orig_node = ox.distance.nearest_nodes(
    G_proj,
    X=oracle_park[1],   # longitude
    Y=oracle_park[0]    # latitude
)

dest_node = ox.distance.nearest_nodes(
    G_proj,
    X=palace_fine_arts[1],
    Y=palace_fine_arts[0]
)

print("Origin node:", orig_node)
print("Destination node:", dest_node)

orig_data = G.nodes[orig_node]
dest_data = G.nodes[dest_node]

print("Origin node (lon,lat):", (orig_data["x"], orig_data["y"]))
print("Dest node (lon,lat):", (dest_data["x"], dest_data["y"]))
print(f"Origin node and Destination node the same? {'Yes' if orig_node == dest_node else 'Nope'}")

Origin node: 370813013
Destination node: 370813013
Origin node (lon,lat): (-122.4865842, 37.7084958)
Dest node (lon,lat): (-122.4865842, 37.7084958)
Origin node and Destination node the same? Yes



## 4) Two search algorithms

### Algorithm A — Dijkstra (Uniform-Cost / Lowest-Cost-First)
- **Why**: With nonnegative edge weights, Dijkstra returns the shortest path by total length.
- **Interpretation**: This is the canonical “GPS-style” algorithm when you minimize total cost. fileciteturn1file1turn1file10

### Algorithm B — A\* with an admissible heuristic
- **Why**: A\* uses both the path cost-so-far and an estimate to the goal:
  \[
  f(p)=g(p)+h(p)
  \]
- **Heuristic**: straight-line (Euclidean) distance between node coordinates in the projected CRS.
  This is admissible because a road path cannot be shorter than the straight-line distance. fileciteturn1file3turn1file4


In [ ]:

# =========================
# 4A) Dijkstra shortest path (distance-minimizing)
# =========================
dijkstra_path = nx.dijkstra_path(G_proj, source=orig_node, target=dest_node, weight="length")
dijkstra_len_m = nx.dijkstra_path_length(G_proj, source=orig_node, target=dest_node, weight="length")

print(f"Dijkstra path nodes: {len(dijkstra_path):,}")
print(f"Dijkstra distance: {dijkstra_len_m:,.1f} meters ({dijkstra_len_m/1609.344:,.3f} miles)")


In [ ]:

# =========================
# 4B) A* shortest path (distance-minimizing with heuristic guidance)
# =========================
def euclidean_heuristic(u, v):
    '''
    Admissible heuristic: straight-line distance between nodes u and v in meters.

    In the projected CRS, each node has coordinates (x, y) in meters.
    The straight-line distance is a lower bound on any drivable road distance.
    '''
    ux, uy = G_proj.nodes[u]["x"], G_proj.nodes[u]["y"]
    vx, vy = G_proj.nodes[v]["x"], G_proj.nodes[v]["y"]
    return math.hypot(vx - ux, vy - uy)

astar_path = nx.astar_path(
    G_proj,
    source=orig_node,
    target=dest_node,
    heuristic=euclidean_heuristic,
    weight="length",
)
astar_len_m = nx.astar_path_length(
    G_proj,
    source=orig_node,
    target=dest_node,
    heuristic=euclidean_heuristic,
    weight="length",
)

print(f"A* path nodes: {len(astar_path):,}")
print(f"A* distance: {astar_len_m:,.1f} meters ({astar_len_m/1609.344:,.3f} miles)")

print("\nDo the algorithms agree on distance?", abs(dijkstra_len_m - astar_len_m) < 1e-6)



## 5) Compare and visualize routes

Because both methods optimize the same nonnegative edge weight (`length`) and the A\* heuristic is admissible,
both should return the same optimal distance. Where A\* can help is *efficiency*: it typically expands fewer
nodes because it focuses search toward the goal. fileciteturn1file6


In [ ]:

# =========================
# 5) Plot graph + route (A* shown; Dijkstra is typically identical here)
# =========================
fig, ax = ox.plot_graph_route(G_proj, astar_path, route_linewidth=4, node_size=0, bgcolor="w")
plt.show()



## 6) Interpretation (what this means)

- The returned node sequence is the ordered set of intersections/road endpoints.
- The distance is the sum of `length` attributes along edges:
  \[
  \text{Route length}=\sum_{e\in \text{route}} \text{length}(e)
  \]
- If A\* and Dijkstra match, that is expected: both are solving the same shortest-path problem, and the A\* heuristic is a safe lower bound.

**Why these algorithms?**  
- Dijkstra is the “gold standard” optimal shortest path under nonnegative weights (uniform-cost search / lowest-cost-first). fileciteturn1file1  
- A\* incorporates heuristic knowledge \(h(n)\) about proximity to the goal to reduce unnecessary exploration while keeping optimality. fileciteturn1file3turn1file4



## 7) Reflection: how I would improve this with more time/knowledge (3–4 paragraphs)

The current solution minimizes **geodesic distance along roads**, which is a clean baseline but not always what a real driver cares about. The most obvious improvement is to optimize **travel time** rather than distance by using a `travel_time` weight, estimated from speed limits and road type. OSMnx can add edge speeds and travel times (using OpenStreetMap attributes where available or reasonable defaults). This changes the cost function from meters to seconds, turning the search objective into the more realistic “fastest route.” It also opens the door to modeling time-of-day variability (rush hour vs. off-peak) and “cost shaping” (e.g., avoiding left turns or preferring arterials).

Second, I would improve realism by incorporating **dynamic or contextual costs** and constraints. For example, we could add penalties for turns (especially left turns across traffic), account for slope (important for biking/walking), or avoid roads with tolls or restricted access. These are all examples of enriching the state/action model: the state might include “current heading” (so turning has cost), vehicle type, or whether a road segment is currently blocked. Conceptually, this is still graph search, but the graph becomes richer (and larger), and costs become multi-factor (distance, time, safety, comfort).

Third, from a search-efficiency perspective, I would implement more advanced routing accelerations used in production navigation systems. Classic options include **bidirectional Dijkstra/A\*** (searching forward from the start and backward from the goal), and preprocessing methods like **contraction hierarchies** or **multi-level overlays**. These methods reduce the practical runtime dramatically on large road networks by exploiting structure (hierarchy of roads) rather than treating the graph as a generic network. This is a direct extension of the course’s theme: the more structure/knowledge you encode (safely), the less “blind” search you need.

Finally, I would upgrade the A\* heuristic beyond pure Euclidean distance. Straight-line distance is admissible but often weak in city grids or when barriers exist (waterfront, parks, one-way systems). More informed (still admissible) heuristics include landmark-based heuristics (ALT: A\*, landmarks, triangle inequality) and heuristics derived from solving simplified problems (e.g., using a coarser graph where each supernode aggregates a neighborhood). The goal would be to increase \(h(n)\) while keeping it admissible, which reduces the number of frontier paths with \(g(p)+h(p)<c\) that must be expanded. fileciteturn1file6



## 8) What to submit

- Export this notebook as **.ipynb** and **.html**. fileciteturn1file8  
- Ensure *all cells are run* and the plotted route + printed distances appear in the outputs.

In Jupyter:
- **File → Save and Checkpoint**
- **File → Download as → HTML**
